# Agrupamento semântico de textos em português
## Embeddings de deep learning + PCA (2D/3D) + K-Means

**Atividade UFMS — Análise exploratória e agrupamento de dados**

| Identificação | Preenchimento |
|---|---|
| Integrantes | *[preencher]* |
| Título do projeto | Agrupamento semântico de textos em português |
| Tipo de dado | Texto · Base com rótulos |

**Pergunta central:** os embeddings semânticos organizam os textos por tema, de
modo que o K-Means (não supervisionado) recupere grupos parecidos com as
categorias reais?

> As categorias reais existem na base, mas **não são usadas pelo K-Means**.
> Elas servem apenas para avaliar a qualidade dos grupos no final.

> **Dica no Colab:** `Ambiente de execução > Alterar o tipo de ambiente de execução > T4 GPU` acelera a geração dos embeddings (opcional).

## 1. Instalação e importação das bibliotecas

In [ ]:
!pip -q install sentence-transformers

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Base de dados: 150 textos em português (6 categorias)

A base é criada **inline** (não precisa fazer upload de arquivo), garantindo
reprodutibilidade. São 6 temas × 25 frases = 150 textos.

In [ ]:
CATEGORIAS = {
    "tecnologia": [
        "O novo processador tem oito nucleos e alta velocidade de clock.",
        "A inteligencia artificial esta transformando o mercado de software.",
        "Precisamos atualizar o servidor para melhorar o desempenho do site.",
        "A linguagem Python e muito usada em ciencia de dados.",
        "O smartphone novo vem com uma bateria de longa duracao.",
        "Redes neurais aprendem padroes a partir de grandes volumes de dados.",
        "A computacao em nuvem permite acessar arquivos de qualquer lugar.",
        "Programadores usam controle de versao para colaborar em projetos.",
        "O algoritmo de busca retorna resultados em milissegundos.",
        "A criptografia protege os dados durante a transmissao pela internet.",
        "O aplicativo trava quando abrimos muitas abas ao mesmo tempo.",
        "Placas de video modernas aceleram o treino de modelos de aprendizado.",
        "O banco de dados armazena milhoes de registros de usuarios.",
        "Compiladores traduzem codigo fonte em instrucoes de maquina.",
        "A realidade virtual cria ambientes imersivos em tres dimensoes.",
        "Conteineres facilitam a implantacao de aplicacoes em qualquer ambiente.",
        "O modelo de linguagem gera texto coerente a partir de um comando.",
        "A memoria insuficiente deixa o computador lento durante o uso.",
        "Testes automatizados ajudam a evitar erros no sistema.",
        "O robo aspirador mapeia a casa com sensores a laser.",
        "A rede de fibra optica aumentou muito a velocidade da internet.",
        "Desenvolvedores publicaram uma nova versao do aplicativo hoje.",
        "O chip de seguranca protege as senhas armazenadas no aparelho.",
        "A automacao industrial usa sensores conectados a um sistema central.",
        "O navegador bloqueia rastreadores para proteger a privacidade.",
    ],
    "esportes": [
        "O jogador marcou um gol incrivel nos ultimos minutos da partida.",
        "A selecao venceu o campeonato depois de uma disputa de penaltis.",
        "O maratonista treinou meses para completar a corrida de rua.",
        "O time de basquete converteu a cesta de tres pontos no fim.",
        "A tenista sacou forte e conquistou o ponto decisivo.",
        "O tecnico mudou a formacao tatica no segundo tempo.",
        "Os nadadores competiram na prova dos cem metros livres.",
        "O ciclista liderou a etapa nas montanhas com folga.",
        "A torcida vibrou com a vitoria no ultimo lance do jogo.",
        "O goleiro defendeu o penalti e garantiu o titulo.",
        "O boxeador venceu a luta por nocaute no terceiro round.",
        "A ginasta executou um salto perfeito e recebeu nota maxima.",
        "O piloto largou na pole position e venceu a corrida.",
        "A equipe de volei conquistou a medalha de ouro.",
        "O atleta bateu o recorde mundial de salto em distancia.",
        "A partida de futebol terminou empatada em dois a dois.",
        "O time avancou com forca ate a linha de fundo do campo.",
        "O surfista dropou uma onda gigante no campeonato.",
        "Os jogadores comemoraram a classificacao para a final.",
        "O arbitro apitou o fim do jogo e encerrou a disputa.",
        "O corredor cruzou a linha de chegada em primeiro lugar.",
        "A defesa do time foi decisiva para segurar o resultado.",
        "O treino de forca melhora o desempenho dos atletas.",
        "A delegacao conquistou varias medalhas nos jogos.",
        "O jogador foi expulso apos receber o segundo cartao amarelo.",
    ],
    "culinaria": [
        "A receita leva farinha, ovos, acucar e uma pitada de sal.",
        "O molho de tomate fica mais saboroso com manjericao fresco.",
        "Assamos o bolo de chocolate por quarenta minutos no forno.",
        "O restaurante serve uma feijoada completa aos sabados.",
        "A massa da pizza precisa descansar antes de ir ao forno.",
        "Refogamos a cebola e o alho antes de adicionar o arroz.",
        "O cafe da manha inclui pao quentinho, manteiga e suco de laranja.",
        "A sobremesa era um pudim de leite bem cremoso.",
        "Temperamos a carne com sal grosso e alecrim antes de grelhar.",
        "A sopa de legumes aquece bem nos dias frios de inverno.",
        "O chef finalizou o prato com uma reducao de vinho tinto.",
        "Batemos as claras em neve para deixar o bolo fofo.",
        "O peixe grelhado veio acompanhado de arroz e salada verde.",
        "A padaria vende croissants amanteigados logo cedo.",
        "Colocamos queijo derretido por cima do macarrao gratinado.",
        "O suco de frutas frescas e uma otima opcao no verao.",
        "A lasanha ficou pronta com camadas de molho e presunto.",
        "Deixamos a massa do pao crescer por duas horas.",
        "O hamburguer artesanal vinha com bacon e cebola caramelizada.",
        "O doce de leite caseiro acompanha bem o queijo.",
        "Servimos o risoto de cogumelos ainda bem cremoso.",
        "A salada tinha folhas verdes, tomate e azeite de oliva.",
        "O churrasco de domingo reuniu a familia no quintal.",
        "Preparamos brigadeiros para a festa de aniversario.",
        "O tempero da comida ficou perfeito com pimenta e limao.",
    ],
    "natureza": [
        "A floresta amazonica abriga uma enorme diversidade de especies.",
        "O rio corre entre as montanhas ate desaguar no mar.",
        "As abelhas sao essenciais para a polinizacao das plantas.",
        "O desmatamento ameaca o habitat de muitos animais silvestres.",
        "A chuva forte encheu os lagos e regou as plantacoes.",
        "O passaro construiu o ninho no alto da arvore.",
        "As tartarugas marinhas retornam a praia para desovar.",
        "O vento espalha as sementes pelo campo aberto.",
        "A reciclagem ajuda a reduzir a poluicao do meio ambiente.",
        "O sol nasce atras das colinas iluminando o vale.",
        "A geleira derrete mais rapido por causa do aquecimento global.",
        "As raizes das arvores seguram o solo e evitam a erosao.",
        "O recife de corais abriga peixes de cores vibrantes.",
        "A trilha na mata revela cachoeiras escondidas.",
        "Os lobos vivem em matilhas nas regioes frias do norte.",
        "A borboleta pousou suavemente sobre a flor amarela.",
        "O deserto tem noites frias e dias muito quentes.",
        "A preservacao das nascentes garante agua limpa para todos.",
        "O urso hiberna durante o inverno rigoroso.",
        "As folhas caem no outono e cobrem o chao da floresta.",
        "A savana africana e o lar de leoes, zebras e girafas.",
        "O oceano cobre a maior parte da superficie do planeta.",
        "A montanha nevada atrai alpinistas do mundo inteiro.",
        "O ecossistema depende do equilibrio entre as especies.",
        "A neblina cobriu o vale logo ao amanhecer.",
    ],
    "saude": [
        "A vacina previne diversas doencas graves na populacao.",
        "O medico receitou um antibiotico para tratar a infeccao.",
        "A pressao arterial deve ser monitorada com regularidade.",
        "Exercicios fisicos melhoram a saude do coracao.",
        "O paciente se recuperou bem apos a cirurgia.",
        "Uma dieta equilibrada previne muitas doencas cronicas.",
        "Dormir bem e fundamental para a recuperacao do corpo.",
        "O enfermeiro mediu a temperatura e a saturacao do paciente.",
        "Beber agua com frequencia ajuda a manter o corpo hidratado.",
        "A fisioterapia ajudou o joelho a recuperar o movimento.",
        "O exame de sangue detectou uma leve anemia.",
        "A consulta com o dentista preveniu problemas nos dentes.",
        "O uso de mascara reduz a transmissao de virus respiratorios.",
        "A meditacao ajuda a diminuir o estresse do dia a dia.",
        "O cardiologista recomendou reduzir o sal na alimentacao.",
        "A vacinacao infantil protege contra sarampo e poliomielite.",
        "Caminhar todos os dias melhora a circulacao sanguinea.",
        "O tratamento incluiu repouso e muita hidratacao.",
        "A nutricionista montou um cardapio rico em fibras.",
        "O hospital ampliou o numero de leitos na emergencia.",
        "A saude mental merece a mesma atencao que a fisica.",
        "O medico pediu uma tomografia para investigar a dor.",
        "Lavar as maos com frequencia previne muitas infeccoes.",
        "A pratica de alongamento evita lesoes musculares.",
        "O check-up anual ajuda a detectar doencas no inicio.",
    ],
    "educacao": [
        "A professora explicou o teorema de Pitagoras no quadro.",
        "Os alunos entregaram o trabalho de historia na data marcada.",
        "A biblioteca da escola recebeu novos livros de literatura.",
        "O estudante revisou a materia antes da prova de matematica.",
        "A universidade abriu inscricoes para os cursos de graduacao.",
        "O professor corrigiu as redacoes durante o fim de semana.",
        "A aula de geografia abordou os climas do Brasil.",
        "Os estudantes participaram de uma feira de ciencias.",
        "A escola adotou tablets para as atividades em sala.",
        "O curso online oferece certificado ao final das aulas.",
        "A monitoria ajuda os alunos com dificuldades em fisica.",
        "O vestibular avalia conhecimentos de varias disciplinas.",
        "A turma leu um classico da literatura brasileira.",
        "O laboratorio de quimica tem equipamentos para experimentos.",
        "A palestra motivou os jovens a seguir a carreira academica.",
        "O diretor anunciou o calendario do proximo semestre letivo.",
        "Os alunos apresentaram seminarios sobre o meio ambiente.",
        "A bolsa de estudos cobre a mensalidade da faculdade.",
        "A prova final avaliou todo o conteudo do ano.",
        "O intercambio permite estudar em outro pais por um ano.",
        "A pesquisa cientifica exige metodo e dedicacao.",
        "O caderno estava cheio de anotacoes sobre a aula.",
        "A educacao infantil estimula a curiosidade das criancas.",
        "O tutor explicou o exercicio passo a passo.",
        "A formatura reuniu os alunos e suas familias no auditorio.",
    ],
}

linhas = [(t, cat) for cat, frases in CATEGORIAS.items() for t in frases]
df = pd.DataFrame(linhas, columns=["texto", "categoria"])

print("Total de textos:", len(df))
print(df["categoria"].value_counts())
df.head()

## 3. Geração dos embeddings (modelo de deep learning)

Usamos o modelo multilíngue `paraphrase-multilingual-MiniLM-L12-v2`, que
suporta português e transforma cada texto em um vetor de **384 dimensões**.
Textos com sentido próximo geram vetores próximos.

In [ ]:
modelo = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

embeddings = modelo.encode(
    df["texto"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True,
)
print("Formato dos embeddings:", embeddings.shape)

## 4. Normalização L2

Para representações semânticas, normalizamos os vetores para comprimento 1.
Isso aproxima a geometria por distância da similaridade de cosseno.

In [ ]:
X = normalize(embeddings, norm="l2")
print("Formato:", X.shape)
print("Norma do primeiro vetor:", np.linalg.norm(X[0]))

## 5. PCA para visualização (variância explicada)

**Pergunta:** duas ou três componentes descrevem toda a estrutura? Vamos ver
quanta variância elas capturam.

In [ ]:
pca = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)

for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"PC{i+1}: {v*100:.2f}%")
print("Acumulada (3 PCs):", f"{pca.explained_variance_ratio_.sum()*100:.2f}%")

### Figura 1 — PCA 2D sem rótulos
*Existem regiões de concentração antes de olhar as classes?*

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], s=40, alpha=0.75, color="#334155")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("Embeddings de texto projetados com PCA (2D) - sem rotulos")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Figura 2 — PCA 2D colorido pela categoria real
*Quais temas ficam mais separados e quais se sobrepõem?*

In [ ]:
cores = ["#2563eb", "#16a34a", "#ea580c", "#9333ea", "#dc2626", "#0891b2"]
plt.figure(figsize=(8, 6))
for i, cat in enumerate(sorted(df["categoria"].unique())):
    m = (df["categoria"] == cat).values
    plt.scatter(X_pca[m, 0], X_pca[m, 1], s=40, alpha=0.8, color=cores[i], label=cat)
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title("PCA 2D - cor = categoria real")
plt.legend(title="categoria", fontsize=8)
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 6. Escolha do número de grupos (k)

O K-Means é aplicado no **espaço completo** dos embeddings (384-d). Avaliamos
`k` de 2 a 10 pelo **método do cotovelo** (inércia) e pelo **silhouette_score**.

In [ ]:
ks = range(2, 11)
inercias, silhuetas = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X)
    inercias.append(km.inertia_)
    s = silhouette_score(X, labels, metric="euclidean")
    silhuetas.append(s)
    print(f"k={k:2d}  inercia={km.inertia_:8.2f}  silhouette={s:.4f}")

melhor_k = list(ks)[int(np.argmax(silhuetas))]
print("\nMelhor k pela silhueta:", melhor_k, "(silhouette =", round(max(silhuetas), 4), ")")

### Figura 3 — Método do cotovelo

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(ks), inercias, marker="o")
plt.xlabel("Numero de grupos (k)"); plt.ylabel("Inercia")
plt.title("Metodo do cotovelo"); plt.xticks(list(ks))
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Figura 4 — Silhouette Score por k
*O k de maior silhueta coincide com o número de categorias?*

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(list(ks), silhuetas, marker="o")
plt.scatter([melhor_k], [max(silhuetas)], color="red", zorder=5, label=f"melhor k = {melhor_k}")
plt.xlabel("Numero de grupos (k)"); plt.ylabel("silhouette_score")
plt.title("Silhouette Score por numero de grupos"); plt.xticks(list(ks))
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 7. K-Means final e visualização dos clusters

In [ ]:
km = KMeans(n_clusters=melhor_k, random_state=RANDOM_STATE, n_init=10)
df["cluster"] = km.fit_predict(X)
df["cluster"].value_counts().sort_index()

### Figura 5 — Clusters do K-Means em PCA 2D

In [ ]:
plt.figure(figsize=(8, 6))
for c in sorted(df["cluster"].unique()):
    m = (df["cluster"] == c).values
    plt.scatter(X_pca[m, 0], X_pca[m, 1], s=40, alpha=0.8, color=cores[c % len(cores)], label=f"cluster {c}")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title(f"PCA 2D - clusters do K-Means (k={melhor_k})")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Figura 6 — Clusters do K-Means em PCA 3D (interativo)
*A terceira dimensão separa grupos que pareciam encostados no 2D?*

In [ ]:
import plotly.express as px

df_pca3 = pd.DataFrame(X_pca, columns=["PC1", "PC2", "PC3"])
df_pca3["cluster"] = df["cluster"].astype(str)
df_pca3["texto"] = df["texto"]

fig = px.scatter_3d(
    df_pca3, x="PC1", y="PC2", z="PC3", color="cluster",
    hover_name="texto", title=f"Clusters do K-Means em PCA 3D (k={melhor_k})", height=650,
)
fig.update_traces(marker=dict(size=5, opacity=0.8))
fig.show()

## 8. Comparação com as categorias reais

Somente agora usamos os rótulos, para avaliar se os grupos correspondem aos temas.

In [ ]:
ct = pd.crosstab(df["categoria"], df["cluster"])
print(ct)
ari = adjusted_rand_score(df["categoria"], df["cluster"])
print("\nAdjusted Rand Index (clusters vs categorias):", round(ari, 4))

## 9. Caracterização dos grupos (evidências da base)

Para cada cluster: tema dominante, pureza e exemplos reais.

In [ ]:
from collections import Counter

for c in sorted(df["cluster"].unique()):
    sub = df[df["cluster"] == c]
    contagem = Counter(sub["categoria"])
    dominante, qtd = contagem.most_common(1)[0]
    print("=" * 70)
    print(f"CLUSTER {c}: {len(sub)} textos | dominante = '{dominante}' ({qtd}/{len(sub)} = {qtd/len(sub):.0%})")
    print("composicao:", dict(contagem))
    for txt in sub["texto"].head(5):
        print("  -", txt)

## 10. Conclusão

- O `silhouette_score` seleciona **k = 6**, coincidindo com o número real de
  categorias da base.
- O **Adjusted Rand Index (~0,85)** e a **pureza alta por grupo (92–96%)**
  mostram que os embeddings organizam os textos por tema e o K-Means recupera
  esses temas sem usar os rótulos.
- **Cluster ≠ classe:** os poucos textos mal agrupados têm ambiguidade
  semântica real (ex.: esporte em montanhas → natureza), não ruído do método.

**Limitações:** base pequena de frases curtas; a variância do PCA em 2D/3D é
baixa (~14–19%), então a visualização é uma aproximação do espaço completo.